# 06 — Xuất báo cáo LaTeX (PDF), Word (.docx) và slide seminar (.pptx)

**Đầu vào:** `GraphK/analysis/` (notebook 05), `GraphK/processed/junyi/`, nội dung `docs/BAO_CAO.md`, `report/noi_dung_slide.json`, `report/thong_tin.json` trong kho mã.
**Đầu ra** (`GraphK/bao_cao/`):

| Tệp | Nội dung |
|---|---|
| `BaoCao_GraphK.pdf`, `latex/` | báo cáo LaTeX (XeLaTeX, phông DejaVu) và mã nguồn |
| `BaoCao_GraphK.docx` | báo cáo Word |
| `Seminar_GraphK.pptx` | slide seminar có case study |
| `kiem_tra/` | ảnh từng trang/slide để rà soát hiển thị |

Không cần GPU. Lần đầu cài TeX Live và LibreOffice mất khoảng 5 phút.

In [ ]:
# Gắn Google Drive, lấy mã nguồn mới nhất của kho GraphK và cài gói graphk
from google.colab import drive
drive.mount('/content/drive')

import os, sys
DRIVE = '/content/drive/MyDrive/GraphK'
REPO = '/content/GraphK'
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists(REPO):
    !git clone -q https://github.com/Trilosphan/GraphK.git {REPO}
else:
    !git -C {REPO} pull -q
%pip install -q -e {REPO}
sys.path.insert(0, f'{REPO}/src')
!git -C {REPO} log --oneline -1

## 1. Cài công cụ

In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y texlive-xetex texlive-latex-extra texlive-fonts-recommended libreoffice-writer libreoffice-impress poppler-utils > /dev/null
!cd {REPO}/report && npm install --silent pptxgenjs docx
!xelatex --version | head -1; node --version; soffice --version

## 2. Gom số liệu và vẽ hình

In [ ]:
from pathlib import Path
ANALYSIS = Path(DRIVE) / 'analysis'
PROC = Path(DRIVE) / 'processed' / 'junyi'
OUT = Path(DRIVE) / 'bao_cao'
OUT.mkdir(parents=True, exist_ok=True)
%cd {REPO}
!python report/gom_ket_qua.py {ANALYSIS} {PROC} report
!python scripts/ve_so_do_kien_truc.py report/figures/hinh1_kien_truc.png
!ls report/figures

## 3. Báo cáo LaTeX → PDF

In [ ]:
!python report/dien_so_lieu.py docs/BAO_CAO.md report/data/ket_qua.json build/BAO_CAO.md
!python report/md_ast.py build/BAO_CAO.md > build/bao_cao.ast.json
!python report/tex.py build/bao_cao.ast.json report/thong_tin.json build/latex/main.tex report/figures
!cd build/latex && xelatex -interaction=nonstopmode -halt-on-error main.tex > /dev/null && xelatex -interaction=nonstopmode main.tex | tail -3
!cp build/latex/main.pdf {OUT}/BaoCao_GraphK.pdf && rm -rf {OUT}/latex && cp -r build/latex {OUT}/latex

## 4. Báo cáo Word và slide seminar

In [ ]:
!NODE_PATH=report/node_modules node report/build_docx.js build/bao_cao.ast.json report/thong_tin.json report/figures {OUT}/BaoCao_GraphK.docx
!NODE_PATH=report/node_modules node report/build_pptx.js report {OUT}/Seminar_GraphK.pptx
# bỏ các mục thư mục trong tệp ZIP: Word/PowerPoint qua dịch vụ đám mây có thể từ chối mở tệp có các mục này
!zip -dq {OUT}/BaoCao_GraphK.docx '*/' ; zip -dq {OUT}/Seminar_GraphK.pptx '*/' ; echo xong

## 5. Kết xuất ảnh để rà soát hiển thị

In [ ]:
CHECK = OUT / 'kiem_tra'
!rm -rf {CHECK} && mkdir -p {CHECK}
!cd {CHECK} && soffice --headless --convert-to pdf {OUT}/BaoCao_GraphK.docx > /dev/null && soffice --headless --convert-to pdf {OUT}/Seminar_GraphK.pptx > /dev/null
!cd {CHECK} && pdftoppm -jpeg -r 60 BaoCao_GraphK.pdf docx && pdftoppm -jpeg -r 80 Seminar_GraphK.pdf slide && pdftoppm -jpeg -r 60 {OUT}/BaoCao_GraphK.pdf tex
!ls {CHECK} | head -80

## 6. Xem nhanh slide

In [ ]:
from IPython.display import Image, display
for f in sorted(CHECK.glob('slide-*.jpg')):
    print(f.name)
    display(Image(filename=str(f), width=900))